# 04 — Statistical feature analysis

This notebook performs the requested univariate statistical analysis for the final 26-feature universe.

Every test uses one saved **inner-training partition**. For a given seed and inner fold, the corresponding outer-test patients and inner-validation patients are excluded. Across 20 seeds and five inner folds, this produces 100 training-side analyses per feature.

The results describe whether each predictor differs among the no-fall, rare-fall, and recurrent-fall groups. They do not measure a feature's added predictive value beside other variables and do not define one global feature subset. The modeling pipeline will refit statistical selection inside its own training folds.

No predictor is imputed, transformed, selected, or used to fit an ML model here.


## 1. Set up paths and libraries

Load the statistical tools and locate the project root explicitly.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

working_directory = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in [working_directory, *working_directory.parents]
        if (folder / "AGENTS.md").is_file()
        and (folder / "docs/research_protocol.md").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Launch this notebook from within the project directory.")

print("Project root:", project_root)


Project root: /Users/rafsan_temp/Library/CloudStorage/OneDrive-SeattleUniversity/SU Projects/pd-fall-risk


Read only the final Notebook 02 inputs and the frozen Notebook 03 assignments. Save this notebook's artifacts in a separate numbered directory.


In [2]:
final_run_directory = (
    project_root
    / "results/final_pipeline/06_final_fit_and_performance_summary"
)
aggregation_directory = final_run_directory / "02_patient_level_aggregation"
split_directory = final_run_directory / "03_setup_and_splits"
output_directory = final_run_directory / "04_statistical_feature_analysis"
output_directory.mkdir(parents=True, exist_ok=True)

required_paths = [
    aggregation_directory / "primary_1040_26_predictors.csv",
    aggregation_directory / "primary_1040_outcome_metadata.csv",
    split_directory / "primary_feature_manifest.csv",
    split_directory / "outer_split_assignments.csv",
    split_directory / "inner_fold_assignments.csv",
]
missing_paths = [path.name for path in required_paths if not path.is_file()]
assert not missing_paths, f"Missing upstream artifacts: {missing_paths}"

print("Inputs ready:", len(required_paths))
print("Outputs:", output_directory.relative_to(project_root))


Inputs ready: 5
Outputs: results/final_pipeline/06_final_fit_and_performance_summary/04_statistical_feature_analysis


## 2. Load the final cohort and split manifests

Align predictors and outcomes by patient number without displaying individual records.


In [3]:
predictors = pd.read_csv(
    aggregation_directory / "primary_1040_26_predictors.csv",
    dtype={"PATNO": "string"},
    low_memory=False,
)
metadata = pd.read_csv(
    aggregation_directory / "primary_1040_outcome_metadata.csv",
    dtype={"PATNO": "string"},
    usecols=["PATNO", "falls_class"],
)
feature_manifest = pd.read_csv(split_directory / "primary_feature_manifest.csv")
outer_assignments = pd.read_csv(
    split_directory / "outer_split_assignments.csv",
    dtype={"PATNO": "string"},
)
inner_assignments = pd.read_csv(
    split_directory / "inner_fold_assignments.csv",
    dtype={"PATNO": "string"},
)

assert predictors["PATNO"].equals(metadata["PATNO"])
assert predictors.columns.tolist() == ["PATNO", *feature_manifest["feature"].tolist()]

data = predictors.merge(metadata, on="PATNO", validate="one_to_one")
features = feature_manifest["feature"].tolist()

print("Patients:", len(data))
print("Features:", len(features))
print(
    "Inner-training partitions:",
    inner_assignments[["split_seed", "inner_validation_fold"]]
    .drop_duplicates()
    .shape[0],
)


Patients: 1040
Features: 26
Inner-training partitions: 100


## 3. Define variable roles and readable names

Nominal variables use contingency-table tests. Ordered scores use Kruskal–Wallis. Continuous or scale variables use ANOVA only when the partition-specific normality and equal-variance checks pass; otherwise they use Kruskal–Wallis.


In [4]:
nominal_features = {
    "DXPOSINS", "DXRIGID", "DOPTHERST", "FEATPOSHYP",
    "ANYFAMPD", "DXTREMOR", "DXBRADY", "DOMSIDE",
}
ordinal_features = {
    "FRZGT12M", "SCAU14", "SCAU16", "NP1SLPD", "NP1URIN",
    "NP3GAIT_COMBINED_MAX", "NP3PSTBL_COMBINED_MAX", "NHY_COMBINED_MAX",
    "NP1CNST",
}
continuous_features = set(features) - nominal_features - ordinal_features

assert nominal_features | ordinal_features | continuous_features == set(features)
assert not (
    nominal_features & ordinal_features
    or nominal_features & continuous_features
    or ordinal_features & continuous_features
)


Attach descriptive labels so the statistical tables remain understandable without referring back to raw PPMI column names.


In [5]:
descriptive_names = {
    "Years_since_PD_diagnosis": "Years since PD diagnosis",
    "Age": "Age at landmark",
    "DXPOSINS": "Postural instability at diagnosis",
    "DXRIGID": "Rigidity at diagnosis",
    "DOPTHERST": "Dopaminergic therapy started",
    "MCATOT": "MoCA total score",
    "FRZGT12M": "Freezing of gait severity",
    "SCAU14": "Lightheaded after standing",
    "SCAU16": "Fainting",
    "GDS_TOTAL": "GDS-15 depression total",
    "NQ_GAUSSIAN_REVISION": "Neuro-QoL Gaussian mobility score (8 items)",
    "NP1RTOT": "MDS-UPDRS Part I rater total",
    "BMI": "Body mass index",
    "NP1SLPD": "Daytime sleepiness",
    "NP1URIN": "Urinary problems",
    "NP3GAIT_COMBINED_MAX": "Part III gait, combined states",
    "NP3PSTBL_COMBINED_MAX": "Part III postural stability, combined states",
    "NHY_COMBINED_MAX": "Hoehn–Yahr stage, combined states",
    "FEATPOSHYP": "Postural hypotension clinical feature",
    "NP3TOT_COMBINED_MAX": "MDS-UPDRS Part III motor total, combined states",
    "NP4TOT": "MDS-UPDRS Part IV complications total",
    "ANYFAMPD": "Family history of Parkinson's disease",
    "DXTREMOR": "Resting tremor at diagnosis",
    "DXBRADY": "Bradykinesia at diagnosis",
    "DOMSIDE": "Predominantly affected side at onset",
    "NP1CNST": "Constipation severity",
}

assert set(descriptive_names) == set(features)


Record Devi's documented test assignments as historical context. The final analysis below does not repeat the historical zero-filling or unadjusted-only procedure.


In [6]:
legacy_kruskal = {
    "Years_since_PD_diagnosis", "MCATOT", "GDS_TOTAL", "NP4TOT",
}
legacy_anova = {"BMI"}
legacy_chi_square = {
    "DXPOSINS", "DXRIGID", "DOPTHERST", "FRZGT12M", "SCAU14", "SCAU16",
    "NP1SLPD", "NP1URIN", "NP3GAIT_COMBINED_MAX", "NP3PSTBL_COMBINED_MAX",
    "NHY_COMBINED_MAX", "FEATPOSHYP", "ANYFAMPD", "DXTREMOR", "DXBRADY",
    "DOMSIDE", "NQMOB37", "NQMOB30", "NQMOB26", "NQMOB32", "NQMOB33",
    "NQMOB31", "NQMOB28",
}


def documented_legacy_test(feature):
    if feature in legacy_kruskal:
        return "kruskal_wallis"
    if feature in legacy_anova:
        return "one_way_anova"
    if feature in legacy_chi_square:
        return "chi_square"
    if feature == "NP1CNST":
        return "not_in_legacy_screen"
    return "not_documented"


variable_roles = pd.DataFrame({
    "feature": features,
    "descriptive_name": [descriptive_names[feature] for feature in features],
    "corrected_role": [
        "nominal" if feature in nominal_features
        else "ordinal" if feature in ordinal_features
        else "continuous_or_scale"
        for feature in features
    ],
    "documented_legacy_test": [documented_legacy_test(feature) for feature in features],
})

display(variable_roles)


,feature,descriptive_name,corrected_role,documented_legacy_test
0,Years_since_PD_diagnosis,Years since PD diagnosis,continuous_or_scale,kruskal_wallis
1,Age,Age at landmark,continuous_or_scale,not_documented
2,DXPOSINS,Postural instability at diagnosis,nominal,chi_square
3,DXRIGID,Rigidity at diagnosis,nominal,chi_square
4,DOPTHERST,Dopaminergic therapy started,nominal,chi_square
5,MCATOT,MoCA total score,continuous_or_scale,kruskal_wallis
6,FRZGT12M,Freezing of gait severity,ordinal,chi_square
7,SCAU14,Lightheaded after standing,ordinal,chi_square
8,SCAU16,Fainting,ordinal,chi_square
9,GDS_TOTAL,GDS-15 depression total,continuous_or_scale,kruskal_wallis


### Why the corrected analysis differs

The historical code documented four Kruskal–Wallis variables, one ANOVA variable, and 23 chi-square variables among the prior 32 variables. Four variables had no documented test, missing values were filled with zero, and p-values were not corrected for testing many features.

The final analysis uses observed values, checks assumptions within each training partition, reports effect sizes, uses permutation p-values for sparse contingency tables, and applies Benjamini–Hochberg false-discovery-rate correction across all 26 features within that partition.


## 4. Define statistical helpers

Benjamini–Hochberg adjustment controls the expected false-discovery proportion among rejected tests within one training partition.


In [7]:
def bh_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan)
    valid = np.flatnonzero(np.isfinite(p_values))

    if not len(valid):
        return adjusted

    order = valid[np.argsort(p_values[valid])]
    ranked = p_values[order] * len(valid) / np.arange(1, len(valid) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted[order] = np.minimum(ranked, 1.0)
    return adjusted


For numeric variables, return the test, assumption diagnostics, and the appropriate effect size: eta-squared for ANOVA or epsilon-squared for Kruskal–Wallis.


In [8]:
def numeric_test(frame, feature, role):
    observed = frame[[feature, "falls_class"]].dropna()
    groups = [
        observed.loc[observed["falls_class"].eq(class_id), feature]
        .astype(float)
        .to_numpy()
        for class_id in [0, 1, 2]
    ]
    counts = [len(group) for group in groups]

    base = {
        "observed_n": len(observed),
        "missing_n": len(frame) - len(observed),
        "normality_pass": np.nan,
        "minimum_normality_p": np.nan,
        "equal_variance_pass": np.nan,
        "equal_variance_p": np.nan,
        "expected_minimum": np.nan,
        "expected_below_five_fraction": np.nan,
        "sparse_expected": np.nan,
    }

    if min(counts) < 2:
        return {
            **base, "test": "insufficient_data", "effect_size_metric": "none",
            "statistic": np.nan, "p_value": np.nan, "effect_size": np.nan,
        }

    combined = np.concatenate(groups)
    if np.unique(combined).size < 2:
        return {
            **base, "test": "no_variation", "effect_size_metric": "none",
            "statistic": 0.0, "p_value": 1.0, "effect_size": 0.0,
        }

    if role == "ordinal":
        method = "kruskal_wallis"
    else:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            normality_p = [
                stats.normaltest(group).pvalue
                if len(group) >= 8 and np.unique(group).size >= 3
                else 0.0
                for group in groups
            ]
            variance_p = stats.levene(*groups, center="median").pvalue

        normality_pass = all(np.isfinite(normality_p)) and min(normality_p) >= 0.05
        equal_variance_pass = np.isfinite(variance_p) and variance_p >= 0.05
        base.update({
            "normality_pass": bool(normality_pass),
            "minimum_normality_p": float(min(normality_p)),
            "equal_variance_pass": bool(equal_variance_pass),
            "equal_variance_p": float(variance_p),
        })
        method = (
            "one_way_anova"
            if normality_pass and equal_variance_pass
            else "kruskal_wallis"
        )

    if method == "one_way_anova":
        result = stats.f_oneway(*groups)
        grand_mean = observed[feature].astype(float).mean()
        between = sum(
            len(group) * (group.mean() - grand_mean) ** 2
            for group in groups
        )
        total = ((observed[feature].astype(float) - grand_mean) ** 2).sum()
        effect_size = between / total if total > 0 else 0.0
        effect_metric = "eta_squared"
    else:
        result = stats.kruskal(*groups)
        effect_size = max(
            0.0,
            (result.statistic - len(groups) + 1) / (len(observed) - len(groups)),
        )
        effect_metric = "epsilon_squared"

    return {
        **base,
        "test": method,
        "effect_size_metric": effect_metric,
        "statistic": float(result.statistic),
        "p_value": float(result.pvalue),
        "effect_size": float(effect_size),
    }


For nominal variables, use chi-square with a deterministic 999-resample permutation p-value when expected counts are sparse. Report bias-corrected Cramér's V.


In [9]:
def categorical_test(frame, feature, random_seed):
    observed = frame[[feature, "falls_class"]].dropna()
    table = pd.crosstab(
        observed[feature],
        observed["falls_class"],
    ).reindex(columns=[0, 1, 2], fill_value=0)
    table = table.loc[table.sum(axis=1).gt(0)]

    base = {
        "observed_n": len(observed),
        "missing_n": len(frame) - len(observed),
        "normality_pass": np.nan,
        "minimum_normality_p": np.nan,
        "equal_variance_pass": np.nan,
        "equal_variance_p": np.nan,
    }

    if table.shape[0] < 2 or (table.sum(axis=0) == 0).any():
        return {
            **base, "test": "no_variation", "effect_size_metric": "none",
            "statistic": 0.0, "p_value": 1.0, "effect_size": 0.0,
            "expected_minimum": np.nan,
            "expected_below_five_fraction": np.nan,
            "sparse_expected": False,
        }

    asymptotic = stats.chi2_contingency(table, correction=False)
    expected = asymptotic.expected_freq
    sparse = bool(expected.min() < 1 or (expected < 5).mean() > 0.20)

    if sparse:
        permutation = stats.PermutationMethod(
            n_resamples=999,
            rng=np.random.default_rng(random_seed),
        )
        result = stats.chi2_contingency(
            table,
            correction=False,
            method=permutation,
        )
        method = "chi_square_permutation"
    else:
        result = asymptotic
        method = "chi_square"

    n = table.to_numpy().sum()
    rows, columns = table.shape
    phi_squared = asymptotic.statistic / n
    corrected_phi = max(
        0.0,
        phi_squared - ((columns - 1) * (rows - 1)) / max(n - 1, 1),
    )
    corrected_rows = rows - ((rows - 1) ** 2) / max(n - 1, 1)
    corrected_columns = columns - ((columns - 1) ** 2) / max(n - 1, 1)
    denominator = max(min(corrected_rows - 1, corrected_columns - 1), 1e-12)
    effect_size = np.sqrt(corrected_phi / denominator)

    return {
        **base,
        "test": method,
        "effect_size_metric": "bias_corrected_cramers_v",
        "statistic": float(result.statistic),
        "p_value": float(result.pvalue),
        "effect_size": float(effect_size),
        "expected_minimum": float(expected.min()),
        "expected_below_five_fraction": float((expected < 5).mean()),
        "sparse_expected": sparse,
    }


Create class-specific descriptive summaries from observed values. Numeric summaries report mean, standard deviation, median, and interquartile range. Nominal summaries retain category counts, including a clearly labeled missing category.


In [10]:
def numeric_group_summary(frame, feature, partition_key):
    rows = []
    for class_id, label in [(0, "no fall"), (1, "rare fall"), (2, "recurrent fall")]:
        group = frame.loc[frame["falls_class"].eq(class_id), feature]
        observed = group.dropna().astype(float)
        rows.append({
            **partition_key,
            "feature": feature,
            "descriptive_name": descriptive_names[feature],
            "falls_class": class_id,
            "class_label": label,
            "class_patients": len(group),
            "observed_n": len(observed),
            "missing_n": int(group.isna().sum()),
            "mean": observed.mean(),
            "standard_deviation": observed.std(ddof=1),
            "median": observed.median(),
            "q1": observed.quantile(0.25),
            "q3": observed.quantile(0.75),
        })
    return rows


def nominal_group_counts(frame, feature, partition_key):
    rows = []
    values = frame[feature].astype("string").fillna("<Missing>")
    for class_id, label in [(0, "no fall"), (1, "rare fall"), (2, "recurrent fall")]:
        class_values = values.loc[frame["falls_class"].eq(class_id)]
        counts = class_values.value_counts(dropna=False).sort_index()
        for category, count in counts.items():
            rows.append({
                **partition_key,
                "feature": feature,
                "descriptive_name": descriptive_names[feature],
                "falls_class": class_id,
                "class_label": label,
                "category": category,
                "patients": int(count),
                "percent_within_class": round(100 * count / len(class_values), 1),
            })
    return rows


## 5. Run the 100 training-side analyses

For each saved inner fold, remove its validation patients from that seed's outer training set. A progress line appears after each seed.


In [11]:
role_lookup = variable_roles.set_index("feature")["corrected_role"].to_dict()
partition_tests = []
missingness_tests = []
numeric_summaries = []
nominal_summaries = []

for split_seed in sorted(inner_assignments["split_seed"].unique()):
    outer_train_ids = set(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(split_seed)
            & outer_assignments["role"].eq("train"),
            "PATNO",
        ]
    )
    outer_test_ids = set(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(split_seed)
            & outer_assignments["role"].eq("test"),
            "PATNO",
        ]
    )

    for inner_fold in range(5):
        validation_ids = set(
            inner_assignments.loc[
                inner_assignments["split_seed"].eq(split_seed)
                & inner_assignments["inner_validation_fold"].eq(inner_fold),
                "PATNO",
            ]
        )
        training_ids = outer_train_ids - validation_ids
        training = data.loc[data["PATNO"].isin(training_ids)].copy()

        assert outer_test_ids.isdisjoint(training_ids)
        assert validation_ids.isdisjoint(training_ids)
        assert len(training) in {582, 583}
        assert training["falls_class"].nunique() == 3

        partition_key = {
            "split_seed": int(split_seed),
            "inner_validation_fold": inner_fold,
            "training_patients": len(training),
        }

        for feature_index, feature in enumerate(features):
            random_seed = 1_000_000 + split_seed * 10_000 + inner_fold * 100 + feature_index
            role = role_lookup[feature]

            if role == "nominal":
                result = categorical_test(training, feature, random_seed)
                nominal_summaries.extend(
                    nominal_group_counts(training, feature, partition_key)
                )
            else:
                result = numeric_test(training, feature, role)
                numeric_summaries.extend(
                    numeric_group_summary(training, feature, partition_key)
                )

            partition_tests.append({
                **partition_key,
                "feature": feature,
                "descriptive_name": descriptive_names[feature],
                "role": role,
                **result,
            })

            missing_frame = pd.DataFrame({
                "missing_status": np.where(
                    training[feature].isna(), "missing", "recorded"
                ),
                "falls_class": training["falls_class"].to_numpy(),
            })
            missing_result = categorical_test(
                missing_frame,
                "missing_status",
                random_seed + 50_000,
            )
            indicator_observed_n = missing_result.pop("observed_n")
            indicator_missing_n = missing_result.pop("missing_n")
            assert indicator_observed_n == len(training) and indicator_missing_n == 0

            source_missing_n = int(training[feature].isna().sum())
            missingness_tests.append({
                **partition_key,
                "feature": feature,
                "descriptive_name": descriptive_names[feature],
                "source_recorded_n": len(training) - source_missing_n,
                "source_missing_n": source_missing_n,
                **missing_result,
            })

    print(f"Completed seed {split_seed:02d}/19: five inner-training partitions")


Completed seed 00/19: five inner-training partitions
Completed seed 01/19: five inner-training partitions
Completed seed 02/19: five inner-training partitions
Completed seed 03/19: five inner-training partitions
Completed seed 04/19: five inner-training partitions
Completed seed 05/19: five inner-training partitions
Completed seed 06/19: five inner-training partitions
Completed seed 07/19: five inner-training partitions
Completed seed 08/19: five inner-training partitions
Completed seed 09/19: five inner-training partitions
Completed seed 10/19: five inner-training partitions
Completed seed 11/19: five inner-training partitions
Completed seed 12/19: five inner-training partitions
Completed seed 13/19: five inner-training partitions
Completed seed 14/19: five inner-training partitions
Completed seed 15/19: five inner-training partitions
Completed seed 16/19: five inner-training partitions
Completed seed 17/19: five inner-training partitions
Completed seed 18/19: five inner-training part

Convert the collected rows to tables and apply FDR correction separately across the 26 feature tests within each training partition.


In [12]:
partition_results = pd.DataFrame(partition_tests)
missingness_results = pd.DataFrame(missingness_tests)
numeric_group_summaries = pd.DataFrame(numeric_summaries)
nominal_group_summaries = pd.DataFrame(nominal_summaries)
partition_columns = ["split_seed", "inner_validation_fold"]

missingness_column_order = [
    "split_seed", "inner_validation_fold", "training_patients", "feature",
    "descriptive_name", "source_recorded_n", "source_missing_n",
    "normality_pass", "minimum_normality_p", "equal_variance_pass",
    "equal_variance_p", "test", "effect_size_metric", "statistic",
    "p_value", "effect_size", "expected_minimum",
    "expected_below_five_fraction", "sparse_expected",
]
missingness_results = missingness_results[missingness_column_order]

for frame in [partition_results, missingness_results]:
    frame["q_value"] = frame.groupby(partition_columns)["p_value"].transform(
        bh_adjust
    )
    frame["fdr_significant"] = frame["q_value"].lt(0.05)

print("Feature tests:", len(partition_results))
print("Missingness tests:", len(missingness_results))
print("Numeric group-summary rows:", len(numeric_group_summaries))
print("Nominal category rows:", len(nominal_group_summaries))


Feature tests: 2600
Missingness tests: 2600
Numeric group-summary rows: 5400
Nominal category rows: 7372


## 6. Summarize test and association stability

A feature may switch between ANOVA and Kruskal–Wallis when assumptions differ across partitions. Summarize each test family separately before creating the readable feature-level table.


In [13]:
test_specific_stability = (
    partition_results.groupby(
        ["feature", "descriptive_name", "test", "effect_size_metric"],
        as_index=False,
    )
    .agg(
        partitions=("p_value", "count"),
        median_p_value=("p_value", "median"),
        median_q_value=("q_value", "median"),
        fdr_significant_partitions=("fdr_significant", "sum"),
        median_effect_size=("effect_size", "median"),
        effect_size_q1=("effect_size", lambda values: values.quantile(0.25)),
        effect_size_q3=("effect_size", lambda values: values.quantile(0.75)),
    )
)
test_specific_stability["partition_fraction"] = (
    test_specific_stability["partitions"] / 100
)


Build one row per feature. The reported effect size is restricted to that feature's most frequently used test, avoiding a mixture of eta-squared and epsilon-squared values.


In [14]:
common_test = (
    test_specific_stability
    .sort_values(["feature", "partitions", "test"], ascending=[True, False, True])
    .groupby("feature", as_index=False)
    .first()[
        ["feature", "test", "effect_size_metric", "partitions", "partition_fraction",
         "median_effect_size", "effect_size_q1", "effect_size_q3"]
    ]
    .rename(columns={
        "test": "most_common_test",
        "effect_size_metric": "effect_size_metric",
        "partitions": "common_test_partitions",
        "partition_fraction": "common_test_fraction",
        "median_effect_size": "common_test_median_effect",
        "effect_size_q1": "common_test_effect_q1",
        "effect_size_q3": "common_test_effect_q3",
    })
)

overall_stability = (
    partition_results.groupby(
        ["feature", "descriptive_name", "role"],
        as_index=False,
    )
    .agg(
        partitions=("p_value", "count"),
        median_observed_n=("observed_n", "median"),
        median_missing_n=("missing_n", "median"),
        median_raw_p=("p_value", "median"),
        median_adjusted_q=("q_value", "median"),
        fdr_significant_partitions=("fdr_significant", "sum"),
    )
)
overall_stability["fdr_significant_fraction"] = (
    overall_stability["fdr_significant_partitions"]
    / overall_stability["partitions"]
)

feature_summary = (
    overall_stability.merge(common_test, on="feature", validate="one_to_one")
    .sort_values(
        ["fdr_significant_fraction", "common_test_median_effect"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)


Summarize missingness associations separately. This asks whether recording availability differs across outcome groups; it does not test the clinical value itself.


In [15]:
missingness_stability = (
    missingness_results.groupby(
        ["feature", "descriptive_name"],
        as_index=False,
    )
    .agg(
        partitions=("p_value", "count"),
        median_source_missing_n=("source_missing_n", "median"),
        median_adjusted_q=("q_value", "median"),
        fdr_significant_partitions=("fdr_significant", "sum"),
        median_effect_size=("effect_size", "median"),
    )
)
missingness_stability["fdr_significant_fraction"] = (
    missingness_stability["fdr_significant_partitions"]
    / missingness_stability["partitions"]
)
missingness_stability = missingness_stability.sort_values(
    ["fdr_significant_fraction", "median_effect_size"],
    ascending=[False, False],
).reset_index(drop=True)


Display the readable 26-row statistical summary and a compact count of association stability. These frequencies describe overlapping training partitions rather than independent replications.


In [16]:
display_columns = [
    "feature", "descriptive_name", "role", "most_common_test",
    "effect_size_metric", "median_observed_n", "median_adjusted_q",
    "fdr_significant_partitions", "fdr_significant_fraction",
    "common_test_median_effect",
]
display(feature_summary[display_columns].round(4))

always_significant = int(feature_summary["fdr_significant_fraction"].eq(1).sum())
usually_significant = int(feature_summary["fdr_significant_fraction"].ge(0.8).sum())
never_significant = int(feature_summary["fdr_significant_fraction"].eq(0).sum())

print(f"FDR-significant in all 100 partitions: {always_significant}/{len(features)}")
print(f"FDR-significant in at least 80 partitions: {usually_significant}/{len(features)}")
print(f"Never FDR-significant: {never_significant}/{len(features)}")


,feature,descriptive_name,role,most_common_test,effect_size_metric,median_observed_n,median_adjusted_q,fdr_significant_partitions,fdr_significant_fraction,common_test_median_effect
0,NQ_GAUSSIAN_REVISION,Neuro-QoL Gaussian mobility score (8 items),continuous_or_scale,kruskal_wallis,epsilon_squared,545.5,0.0000,100,1.00,0.1561
1,FRZGT12M,Freezing of gait severity,ordinal,kruskal_wallis,epsilon_squared,546.0,0.0000,100,1.00,0.1528
2,NHY_COMBINED_MAX,"Hoehn–Yahr stage, combined states",ordinal,kruskal_wallis,epsilon_squared,582.0,0.0000,100,1.00,0.1496
3,NP3GAIT_COMBINED_MAX,"Part III gait, combined states",ordinal,kruskal_wallis,epsilon_squared,582.0,0.0000,100,1.00,0.1319
4,NP3PSTBL_COMBINED_MAX,"Part III postural stability, combined states",ordinal,kruskal_wallis,epsilon_squared,582.0,0.0000,100,1.00,0.1130
5,Years_since_PD_diagnosis,Years since PD diagnosis,continuous_or_scale,kruskal_wallis,epsilon_squared,582.0,0.0000,100,1.00,0.1093
6,NP4TOT,MDS-UPDRS Part IV complications total,continuous_or_scale,kruskal_wallis,epsilon_squared,456.5,0.0000,100,1.00,0.1087
7,NP1RTOT,MDS-UPDRS Part I rater total,continuous_or_scale,kruskal_wallis,epsilon_squared,582.0,0.0000,100,1.00,0.1028
8,NP3TOT_COMBINED_MAX,"MDS-UPDRS Part III motor total, combined states",continuous_or_scale,kruskal_wallis,epsilon_squared,582.0,0.0000,100,1.00,0.0924
9,NP1CNST,Constipation severity,ordinal,kruskal_wallis,epsilon_squared,582.0,0.0000,100,1.00,0.0914


FDR-significant in all 100 partitions: 16/26
FDR-significant in at least 80 partitions: 17/26
Never FDR-significant: 2/26


Show only missingness patterns that were FDR-significant in at least one training partition.


In [17]:
associated_missingness = missingness_stability.loc[
    missingness_stability["fdr_significant_partitions"].gt(0)
]

display(
    associated_missingness[
        ["feature", "descriptive_name", "median_source_missing_n",
         "fdr_significant_partitions", "fdr_significant_fraction",
         "median_effect_size"]
    ].round(4)
)


,feature,descriptive_name,median_source_missing_n,fdr_significant_partitions,fdr_significant_fraction,median_effect_size
0,NP4TOT,MDS-UPDRS Part IV complications total,126.0,98,0.98,0.1768
1,ANYFAMPD,Family history of Parkinson's disease,34.5,27,0.27,0.0980
2,DOPTHERST,Dopaminergic therapy started,113.5,20,0.20,0.0935
3,NQ_GAUSSIAN_REVISION,Neuro-QoL Gaussian mobility score (8 items),37.0,10,0.10,0.0529
4,FRZGT12M,Freezing of gait severity,36.0,10,0.10,0.0492


## 7. Validate the analysis

Confirm that all 26 variables were tested in every one of the 100 training partitions and that observed plus missing counts always equal the partition size.


In [18]:
validation_rows = []


def check(name, condition, detail):
    validation_rows.append({
        "check": name,
        "passed": bool(condition),
        "detail": detail,
    })


partition_counts = partition_results.groupby(partition_columns).size()
feature_counts = partition_results.groupby("feature").size()
count_identity = (
    partition_results["observed_n"] + partition_results["missing_n"]
).eq(partition_results["training_patients"])

check("26-feature contract", len(features) == 26, "expected 26")
check("100 training partitions", len(partition_counts) == 100, "20 seeds × 5 folds")
check("26 tests per partition", partition_counts.eq(26).all(), "expected 26")
check("100 tests per feature", feature_counts.eq(100).all(), "expected 100")
check(
    "training partition sizes",
    partition_results["training_patients"].isin([582, 583]).all(),
    "outer training minus one inner validation fold",
)
check("observed plus missing identity", count_identity.all(), "equals training patients")
check(
    "valid raw p-values",
    partition_results["p_value"].between(0, 1).all(),
    "all feature tests",
)
check(
    "valid adjusted q-values",
    partition_results["q_value"].between(0, 1).all(),
    "within-partition BH adjustment",
)
check(
    "nonnegative effect sizes",
    partition_results["effect_size"].ge(0).all(),
    "all feature tests",
)
check(
    "missingness coverage",
    len(missingness_results) == 2_600,
    "26 features × 100 partitions",
)
check(
    "source missing-count identity",
    (
        missingness_results["source_recorded_n"]
        + missingness_results["source_missing_n"]
    ).eq(missingness_results["training_patients"]).all(),
    "recorded plus missing equals training patients",
)
check(
    "26-row feature summary",
    len(feature_summary) == 26 and feature_summary["feature"].is_unique,
    "one row per candidate feature",
)
check(
    "constipation included",
    feature_summary["feature"].eq("NP1CNST").sum() == 1,
    "new candidate analyzed",
)

validation = pd.DataFrame(validation_rows)
display(validation)
assert validation["passed"].all(), validation.loc[~validation["passed"]]
print(f"Validation checks passed: {validation['passed'].sum()}/{len(validation)}")


,check,passed,detail
0,26-feature contract,True,expected 26
1,100 training partitions,True,20 seeds × 5 folds
2,26 tests per partition,True,expected 26
3,100 tests per feature,True,expected 100
4,training partition sizes,True,outer training minus one inner validation fold
5,observed plus missing identity,True,equals training patients
6,valid raw p-values,True,all feature tests
7,valid adjusted q-values,True,within-partition BH adjustment
8,nonnegative effect sizes,True,all feature tests
9,missingness coverage,True,26 features × 100 partitions


Validation checks passed: 13/13


## 8. Save the statistical artifacts

Write new files or verify that existing files are identical. Differing results are never overwritten silently.


In [19]:
def frames_equivalent(current, existing):
    if current.columns.tolist() != existing.columns.tolist():
        return False
    if current.shape != existing.shape:
        return False

    for column in current.columns:
        left = current[column]
        right = existing[column]
        left_numeric = pd.to_numeric(left, errors="coerce")
        right_numeric = pd.to_numeric(right, errors="coerce")
        left_numeric_ok = left_numeric.notna().eq(left.notna()).all()
        right_numeric_ok = right_numeric.notna().eq(right.notna()).all()

        if left_numeric_ok and right_numeric_ok:
            if not np.allclose(
                left_numeric.to_numpy(dtype=float),
                right_numeric.to_numpy(dtype=float),
                rtol=1e-12,
                atol=1e-12,
                equal_nan=True,
            ):
                return False
        else:
            left_text = left.astype("string").fillna("<NA>")
            right_text = right.astype("string").fillna("<NA>")
            if not left_text.reset_index(drop=True).equals(
                right_text.reset_index(drop=True)
            ):
                return False

    return True


def save_new_or_equivalent(frame, path):
    if path.is_file():
        existing = pd.read_csv(path, low_memory=False)
        if not frames_equivalent(frame.reset_index(drop=True), existing):
            raise FileExistsError(
                f"Existing artifact truly differs and was not overwritten: {path.name}"
            )
        return "already equivalent"

    frame.to_csv(path, index=False)
    return "created"


artifacts = {
    "statistical_variable_roles.csv": variable_roles,
    "inner_training_feature_tests.csv": partition_results,
    "numeric_group_summaries.csv": numeric_group_summaries,
    "nominal_group_counts.csv": nominal_group_summaries,
    "test_specific_stability.csv": test_specific_stability,
    "feature_statistical_summary.csv": feature_summary,
    "missingness_inner_training_tests.csv": missingness_results,
    "missingness_statistical_stability.csv": missingness_stability,
    "statistical_validation.csv": validation,
}

save_rows = []
for filename, frame in artifacts.items():
    status = save_new_or_equivalent(frame, output_directory / filename)
    save_rows.append({"artifact": filename, "status": status, "rows": len(frame)})

save_summary = pd.DataFrame(save_rows)
display(save_summary)
print(f"Validation checks passed: {validation['passed'].sum()}/{len(validation)}")
print("No global feature subset was created and no ML model was fitted.")


,artifact,status,rows
0,statistical_variable_roles.csv,created,26
1,inner_training_feature_tests.csv,created,2600
2,numeric_group_summaries.csv,created,5400
3,nominal_group_counts.csv,created,7372
4,test_specific_stability.csv,created,28
5,feature_statistical_summary.csv,created,26
6,missingness_inner_training_tests.csv,created,2600
7,missingness_statistical_stability.csv,created,26
8,statistical_validation.csv,created,13


Validation checks passed: 13/13
No global feature subset was created and no ML model was fitted.


## How to interpret the result

A low adjusted q-value means the feature's observed distribution differed among the three fall groups in that training partition after accounting for 26 simultaneous feature tests. It does not show:

- which pair of fall groups is responsible without additional post-hoc testing;
- whether the association is causal;
- whether the variable improves prediction after the other features are included;
- whether it specifically solves rare-faller discrimination.

Notebook 05 will compare no removal, corrected FDR selection, L1 selection, and Extra-Trees selection inside the saved training pipeline. The pooled stability table from this notebook is for reporting and cannot supply one feature list to every split.
